Extracting API

In [0]:
import getpass
import requests
from pyspark.sql.functions import schema_of_json, lit
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType
import json
import re
import time

In [0]:
path_output = getpass.getpass("Path output to save the tables: ") 
url = getpass.getpass("Api url: ") 
user = getpass.getpass("Api username: ") 
password = getpass.getpass("Api password: ") 

In [0]:
endpoint = 'SalesOrderHeader'
def get_first_row_api(endpoint):
    try:
        response = requests.get(
            url + endpoint, 
            params={'offset': 0, 'limit': 1}, 
            auth=(user, password)
        )
        return response.json()
    except Exception as e:
        print(f'Error to get response: {e}')
        return None

def get_schema_api(response):
    try:
        first_row = json.dumps(response[0])
        first_row_data = response[0]
        json_schema = schema_of_json(lit(first_row))
        
        date_fields = [
            key for key, value in first_row_data.items()
            if isinstance(value, str) and re.match(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}', value)
        ]
        
        if date_fields:        
            base_schema_df = spark.range(1).select(json_schema.alias("base_schema"))
            base_schema_string = base_schema_df.collect()[0]['base_schema']
            
            get_date_types = base_schema_string
            for date_field in date_fields:
                get_date_types = get_date_types.replace(
                    f"{date_field}: STRING", 
                    f"{date_field}: TIMESTAMP"
                )
            return get_date_types
        else:
            schema_df = spark.range(1).select(json_schema.alias("schema"))
            return schema_df.collect()[0]['schema']
        
    except Exception as e:
        print(f'Error getting schema: {e}')
        return None

def get_api_data(endpoint, offset, max_retries=60, limit=100000):
    print(f'Offset: {offset}')
    
    for attempt in range(max_retries):
        try:
            response = requests.get(
                url + endpoint, 
                params={'offset': offset, 'limit': limit}, 
                auth=(user, password),
                timeout=30
            )
            response.raise_for_status()
            print(f'Offset {offset} completed')
            return response.json()
            
        except requests.exceptions.Timeout:
            print(f'Timeout {endpoint} offset {offset} - attempt {attempt + 1}/{max_retries}')
            if attempt < max_retries - 1:
                time.sleep(3)
            else:
                return None
                
        except Exception as e:
            print(f'Error {endpoint} offset {offset}: {str(e)}')
            return None
    return None

def create_spark_df(results, schema_df):
    try:
        all_data = []
        for result in results:
            if 'data' in result and result['data']:
                all_data.extend(result['data'])
            
        if isinstance(schema_df, str):
            json_strings = [json.dumps(row) for row in all_data]
            temp_df = spark.createDataFrame([(s,) for s in json_strings], ["json_str"])
            
            df = temp_df.select(from_json("json_str", schema_df).alias("data")).select("data.*")
            return df
        else:
            df = spark.createDataFrame(all_data)
            return df
            
    except Exception as e:
        print(f'Error creating df: {e}')
        return None

def el_data_api():
    response = get_first_row_api(endpoint)

    schema_json = get_schema_api(response['data'])
    
    total_rows = response['total']
    api_data = []
    for offset in range(0, total_rows, 100000):
        api_data.append(get_api_data(endpoint, offset))
    
    df = create_spark_df(api_data, schema_json)
    display(df)

el_data_api()